# Exp 2 — fruit still life: Gemini app vs Interactions API (Nano Banana 2)

Re-runs the exp 2 edit chain with Nano Banana 2 (`gemini-3.1-flash-image`) through the
multi-turn Interactions API — high thinking level, one chain per output size (2K and 1K).
Each chain starts from `data/exp2/0.jpeg` (the image generated in the Gemini app) and
applies the same 10 edits, each turn building on the previous interaction. Outputs are
saved to `data/exp2_api_nb2_2k/` and `data/exp2_api_nb2_1k/`; chains whose outputs are
already on disk are skipped, so re-running the notebook costs no API calls.

In [1]:
import base64
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GOOGLE_GEMINI_API_KEY"])

MODEL = "gemini-3.1-flash-image"  # Nano Banana 2
GENERATION_CONFIG = {"thinking_level": "high"}
SIZES = ["2K", "1K"]

APP_DIR = Path("../data/exp2")  # Gemini app outputs: 0.jpeg (start) ... 10.jpeg
# Interactions API outputs, same numbering, one directory per output size.
API_DIRS = {size: Path(f"../data/exp2_api_nb2_{size.lower()}") for size in SIZES}

In [2]:
EDITS = [
    'Add a single ripe yellow banana on the table to the left of the red apple. Keep the apple unchanged and preserve the white background, table, lighting, camera angle, shadows, and overall composition exactly as they are.',
    'Add a single realistic orange on the table to the right of the red apple. Preserve the apple, the banana, and everything else exactly as they are, including the table, white background, lighting, shadows, and camera angle.',
    'Add a single green pear behind the red apple, slightly offset so it remains visible. Preserve the apple, banana, orange, and all other elements exactly as they are. Keep the same white background, table, lighting, shadows, and composition.',
    'Add a single yellow lemon in front of the red apple, leaving all existing fruits visible. Preserve the apple, banana, orange, pear, and everything else exactly as they are. Keep the white background, table, lighting, shadows, and camera angle unchanged.',
    'Add a single peach on the front-left area of the table, near the banana but not overlapping it. Preserve all existing fruits and all other elements exactly as they are, including the table, background, lighting, shadows, and composition.',
    'Add a single dark purple plum on the front-right area of the table, near the orange but not overlapping it. Preserve all existing fruits and the rest of the image exactly as they are.',
    'Add a small bunch of green grapes on the back-right area of the table, positioned so the grapes are fully visible. Preserve all previously existing fruits and keep the white background, table, lighting, shadows, and camera angle unchanged.',
    'Add a single large strawberry on the table near the lemon in the front-center area. Preserve all existing fruits and the rest of the image exactly as they are.',
    'Add a whole brown kiwi on the table near the peach on the left side. Preserve all other fruits and all other image elements exactly as they are, including lighting, shadows, background, and composition.',
    'Add a single mango on the back-left area of the table, balancing the composition while keeping all fruits visible. Preserve every previously added fruit and everything else in the image exactly as they are. Keep the same white background, table, lighting, shadows, and camera angle.',
]

In [3]:
import time


def create_with_retry(**kwargs):
    # A transient connection reset would kill the whole chain — retry with backoff.
    for attempt in range(1, 6):
        try:
            return client.interactions.create(**kwargs)
        except Exception as exc:
            if attempt == 5:
                raise
            wait = 15 * attempt
            print(f"  attempt {attempt} failed ({type(exc).__name__}: {exc}); retrying in {wait}s")
            time.sleep(wait)


def run_chain(size):
    api_dir = API_DIRS[size]
    api_dir.mkdir(exist_ok=True)

    def save_output(turn, interaction):
        assert interaction.output_image, f"{size} turn {turn} returned no image"
        (api_dir / f"{turn}.jpeg").write_bytes(base64.b64decode(interaction.output_image.data))
        print(f"{size} turn {turn:2d} done  ({interaction.id})")

    # Turn 0 is the image generated in the Gemini app, shared by all chains.
    shutil.copy(APP_DIR / "0.jpeg", api_dir / "0.jpeg")

    # Each turn chains on the previous interaction id, so a partial chain can't be
    # resumed — skip the API entirely when every turn is already on disk.
    if all((api_dir / f"{turn}.jpeg").exists() for turn in range(1, len(EDITS) + 1)):
        print(f"{size}: all {len(EDITS)} turns already in {api_dir} — skipping API calls")
        return

    response_format = {"type": "image", "mime_type": "image/jpeg", "aspect_ratio": "16:9", "image_size": size}
    start_image = base64.b64encode((APP_DIR / "0.jpeg").read_bytes()).decode()

    # Turn 1 sends the starting image; every later turn chains on the previous
    # interaction, so the model keeps the full editing history as context.
    interaction = create_with_retry(
        model=MODEL,
        input=[
            {"type": "text", "text": EDITS[0]},
            {"type": "image", "data": start_image, "mime_type": "image/jpeg"},
        ],
        response_format=response_format,
        generation_config=GENERATION_CONFIG,
    )
    save_output(1, interaction)

    for turn, prompt in enumerate(EDITS[1:], start=2):
        interaction = create_with_retry(
            model=MODEL,
            input=prompt,
            previous_interaction_id=interaction.id,
            response_format=response_format,
            generation_config=GENERATION_CONFIG,
        )
        save_output(turn, interaction)


for size in SIZES:
    run_chain(size)

2K: all 10 turns already in ../data/exp2_api_nb2_2k — skipping API calls


1K turn  1 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXT1NTR2F2V2FKUHkya2RVUGlvZWg4QXM)


1K turn  2 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXVUNTR2FwX25DTExsa2RVUDdvSzZnUTg)


1K turn  3 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXWkNTR2FyZk1BZEhtbnNFUC0tWFk4UWc)


1K turn  4 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXZVNTR2F0bTVCWWJobnNFUDRQcWIwUVk)


...

1K turn  6 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXbmlTR2FxZjdJWjNubnNFUDM0eWc4UWc)


1K turn  7 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXdHlTR2FwYS1Kb0NGa2RVUGxxaTl3UWc)


1K turn  8 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXMUNTR2F0X3VOX1hpbnNFUHVzU1owUVU)


1K turn  9 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXOENTR2FwU1hMckdxa2RVUHR1anUyQWM)


1K turn 10 done  (v1_ChdPU1NHYXZXYUpQeTJrZFVQaW9laDhBcxIXQ1NXR2FzSHRNSmY0bnNFUHFNZXBzUTA)


## Comparison

One widget per output size. Pick the turn with the top slider, then drag the divider
on the image: left of the line is the **Gemini app** result, right is the **API** result.

In [4]:
from IPython.display import Markdown, display

from nanobanana.compare import comparison

for size in SIZES:
    display(Markdown(f"### Gemini app vs API — Nano Banana 2, {size}"))
    display(comparison(APP_DIR, API_DIRS[size], EDITS))

### Gemini app vs API — Nano Banana 2, 2K

### Gemini app vs API — Nano Banana 2, 1K